# Integracao de Dados para Previsao de `total_incoming` em Multiplos POIs

Grupo 2

João Costa | 113564

Sílvia Ribeiro | 129428

Vitória Rodrigues | 130557

Este notebook apresenta o processo de integracao de dados desenvolvido para suportar as etapas seguintes de **pre-processamento** e **analise exploratoria dos dados (EDA)**, utilizando como tabela alvo a `cagg_node_incoming`.

Nesta versao final, o dataset e construido apenas com as tabelas consideradas essenciais para o projeto:

- `cagg_node_incoming`
- `DIM_DATE`
- `DIM_GEOGRAPHY`
- `metereologia_openmeteo_limpa_diaria`

## Objetivo

Construir um unico dataframe diario, ao nivel de:

- `movement_date`
- `geography_key`

com a variavel alvo:

- `total_incoming`

## POIs considerados

Os Pontos de Interesse (POIs) definidos para esta integracao sao:

- Sete Cidades - Ponte entre Lagoas
- Vista do Rei - Miradouro
- Lagoa do Canario
- Caldeiras das Furnas (Aglomerado Urbano)
- Grota do Inferno

## Resultado esperado

O dataframe final `df_final` fica preparado para ser utilizado nas etapas subsequentes de:

- pre-processamento
- analise exploratoria dos dados


In [ ]:

import pandas as pd
import numpy as np
from IPython.display import display

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)
pd.set_option("display.max_colwidth", 120)


## 1. Carregamento dos datasets

Os dados sao carregados a partir da tabela alvo agregada, das dimensoes auxiliares e da tabela meteorologica diaria, previamente limpa e preparada a partir do Open-Meteo.

In [ ]:
import pandas as pd
from pathlib import Path
from IPython.display import display

START_DIR = Path.cwd().resolve()
print(f"Diretoria atual de execução: {START_DIR}")


DATA_DIR = None
for root in [START_DIR, *START_DIR.parents]:
    candidate = root / "dataset"
    if candidate.exists() and candidate.is_dir():
        DATA_DIR = candidate.resolve()
        break

if DATA_DIR is None:
    raise FileNotFoundError(
        "Não foi possível localizar a pasta central 'dataset'. "
        "Certifica-te de que criaste uma pasta chamada 'dataset' na raiz do projeto com os CSVs."
    )

print(f"Pasta unificada de dados encontrada em: {DATA_DIR}")


DATA_PATHS = {
    "cagg_node_incoming": DATA_DIR / "cagg_node_incoming.csv",
    "dim_date": DATA_DIR / "dim_date.csv",
    "dim_geography": DATA_DIR / "dim_geography.csv",
    "weather_daily": DATA_DIR / "metereologia_openmeteo_limpa_diaria.csv",
}

for name, file_path in DATA_PATHS.items():
    if not file_path.exists():
        raise FileNotFoundError(f"Ficheiro obrigatório não encontrado para {name}: {file_path}")


cagg_node_incoming = pd.read_csv(DATA_PATHS["cagg_node_incoming"], low_memory=False)
dim_date = pd.read_csv(DATA_PATHS["dim_date"], low_memory=False)
dim_geography = pd.read_csv(DATA_PATHS["dim_geography"], low_memory=False)
weather_daily = pd.read_csv(DATA_PATHS["weather_daily"], low_memory=False)

loaded_tables = {
    "cagg_node_incoming": cagg_node_incoming,
    "dim_date": dim_date,
    "dim_geography": dim_geography,
    "weather_daily": weather_daily,
}

display(pd.DataFrame(
    {
        "table": loaded_tables.keys(),
        "rows": [df.shape[0] for df in loaded_tables.values()],
        "columns": [df.shape[1] for df in loaded_tables.values()],
    }
))

In [ ]:

def inspect_dataframe(df, name, head_rows=5):
    print(f"{name} - shape: {df.shape}")
    display(df.head(head_rows))
    df.info()
    print("\nMissing values:")
    display(df.isna().sum().sort_values(ascending=False).head(15).to_frame("missing_values"))


## 2. Definicao do conjunto fixo de POIs

Os POIs utilizados no projeto sao definidos manualmente. Esta lista garante consistencia entre a fase de integracao, o pre-processamento e a analise exploratoria.

In [ ]:

TARGET_POIS = {
    9246: "Sete Cidades - Ponte entre Lagoas",
    10026: "Vista do Rei - Miradouro",
    7175: "Lagoa do Canario",
    6967: "Caldeiras das Furnas (Aglomerado Urbano)",
    7110: "Grota do Inferno",
}

selected_pois_df = pd.DataFrame(
    {
        "geography_key": list(TARGET_POIS.keys()),
        "requested_poi_name": list(TARGET_POIS.values()),
    }
)

selected_pois_df


## 3. Inspecao inicial da tabela alvo

A `cagg_node_incoming` constitui a base principal do dataset. Antes da realizacao dos merges, a tabela e inspecionada para validar a sua granularidade, as colunas disponiveis e a qualidade geral dos dados.

In [ ]:

inspect_dataframe(cagg_node_incoming, "CAGG_NODE_INCOMING")


## 4. Construcao da target diaria

Uma vez que o objetivo do projeto e trabalhar com um dataset diario, a tabela alvo e agregada para o nivel:

- `movement_date`
- `geography_key`

A variavel alvo final passa a ser a soma diaria de `total_incoming`.

In [ ]:

cagg_node_incoming["movement_date"] = pd.to_datetime(cagg_node_incoming["movement_date"], errors="coerce")
cagg_node_incoming["time_bucket"] = pd.to_datetime(cagg_node_incoming["time_bucket"], errors="coerce")

incoming_daily = (
    cagg_node_incoming.groupby(["movement_date", "geography_key"], as_index=False)
    .agg(
        total_incoming=("total_incoming", "sum"),
        incoming_record_count=("record_count", "sum"),
        incoming_source_rows=("total_incoming", "size"),
    )
)

incoming_daily["date_key"] = incoming_daily["movement_date"].dt.strftime("%Y%m%d").astype(int)

inspect_dataframe(incoming_daily, "Target diaria agregada")
print("Duplicados por movement_date + geography_key:", int(incoming_daily.duplicated(["movement_date", "geography_key"]).sum()))


## 5. Verificacao da disponibilidade dos POIs definidos

Nem todos os POIs definidos inicialmente estao necessariamente presentes no subconjunto da tabela alvo disponibilizado. Por esse motivo, o notebook verifica quais os POIs com dados observados e quais os que ficam sem cobertura nesta versao do dataset.

In [ ]:

poi_availability = (
    selected_pois_df.merge(
        dim_geography[["geography_key", "geography_name", "poi_name", "poi_category", "timezone", "is_sensorized"]],
        on="geography_key",
        how="left",
    )
    .merge(
        incoming_daily.groupby("geography_key").agg(
            available_days=("movement_date", "nunique"),
            total_incoming_sum=("total_incoming", "sum"),
        ).reset_index(),
        on="geography_key",
        how="left",
    )
)

poi_availability["has_target_data"] = poi_availability["available_days"].notna()
poi_availability = poi_availability.sort_values(["has_target_data", "available_days"], ascending=[False, False])
poi_availability


## 6. Filtragem da target para os POIs com dados disponiveis

A integracao principal prossegue apenas com os POIs que apresentam observacoes reais na `cagg_node_incoming`. Esta decisao e mantida explicita ao longo do notebook para assegurar transparencia metodologica.

In [ ]:

available_poi_keys = poi_availability.loc[poi_availability["has_target_data"], "geography_key"].tolist()
missing_poi_keys = poi_availability.loc[~poi_availability["has_target_data"], "geography_key"].tolist()

print("POIs com dados na target:", available_poi_keys)
print("POIs sem dados na target:", missing_poi_keys)

poi_target = (
    incoming_daily.loc[incoming_daily["geography_key"].isin(available_poi_keys)]
    .copy()
    .sort_values(["geography_key", "movement_date"])
    .reset_index(drop=True)
)

inspect_dataframe(poi_target, "Serie diaria da target para os POIs selecionados")


## 7. Resumo inicial da target por POI

Este resumo permite observar a cobertura temporal e o volume agregado da variavel alvo em cada POI antes da integracao com as tabelas auxiliares.

In [ ]:

poi_target_summary = (
    poi_target.groupby("geography_key")
    .agg(
        available_days=("movement_date", "nunique"),
        total_incoming_sum=("total_incoming", "sum"),
        avg_total_incoming=("total_incoming", "mean"),
        min_date=("movement_date", "min"),
        max_date=("movement_date", "max"),
    )
    .reset_index()
    .merge(
        dim_geography[["geography_key", "poi_name", "geography_name", "poi_category"]],
        on="geography_key",
        how="left",
    )
    .sort_values(["available_days", "total_incoming_sum"], ascending=[False, False])
)
poi_target_summary


## 8. Integracao da `DIM_DATE`

A dimensao temporal acrescenta variaveis de calendario uteis para as etapas seguintes de pre-processamento e analise exploratoria.

In [ ]:

date_columns = [
    "date_key",
    "date_value",
    "day_of_week",
    "day_name",
    "day_of_month",
    "day_of_year",
    "week_of_year",
    "month_number",
    "month_name",
    "quarter_number",
    "year_number",
    "season",
    "is_weekend",
    "is_holiday",
    "is_peak_season",
    "holiday_name",
    "is_business_day",
]

df_step_date = poi_target.merge(dim_date[date_columns], on="date_key", how="left", validate="m:1")
inspect_dataframe(df_step_date, "Depois do merge com DIM_DATE")


## 9. Integracao da `DIM_GEOGRAPHY`

A dimensao geografica adiciona contexto descritivo aos POIs selecionados, permitindo enriquecer o dataset com informacao espacial e classificatoria.

In [ ]:

geography_columns = [
    "geography_key",
    "geography_type",
    "geography_name",
    "poi_id",
    "poi_name",
    "poi_category",
    "island_name",
    "concelho_name",
    "freguesia_name",
    "latitude",
    "longitude",
    "timezone",
    "is_sensorized",
]

df_step_geography = df_step_date.merge(
    dim_geography[geography_columns],
    on="geography_key",
    how="left",
    validate="m:1",
)
inspect_dataframe(df_step_geography, "Depois do merge com DIM_GEOGRAPHY")


## 10. Inspecao e preparacao da meteorologia diaria

A tabela `metereologia_openmeteo_limpa_diaria` foi previamente limpa e agregada ao nivel diario. Como representa o contexto meteorologico da ilha de Sao Miguel, as variaveis meteorologicas sao integradas por `movement_date`, ficando comuns a todos os POIs presentes na mesma data.

In [ ]:

weather_daily["movement_date"] = pd.to_datetime(weather_daily["movement_date"], errors="coerce")
inspect_dataframe(weather_daily, "Meteorologia diaria limpa")


## 11. Validacao do overlap temporal com a meteorologia

Antes da integracao da meteorologia, e realizada uma verificacao simples para confirmar se a tabela meteorologica cobre efetivamente as datas presentes na target dos POIs.

In [ ]:

poi_dates = set(df_step_geography["movement_date"].dropna())
weather_dates = set(weather_daily["movement_date"].dropna())
weather_overlap = len(poi_dates & weather_dates)

print("Datas da target:", len(poi_dates))
print("Datas da meteorologia:", len(weather_dates))
print("Overlap temporal:", weather_overlap)


## 12. Integracao da meteorologia diaria no dataset principal

Uma vez que os dados meteorologicos representam Sao Miguel como um todo, o merge e realizado por `movement_date`. Desta forma, o contexto meteorologico diario fica associado a todos os POIs presentes na respetiva data.

In [ ]:

weather_columns = [
    "movement_date",
    "weather_hourly_records",
    "weather_temperature_2m_mean",
    "weather_temperature_2m_min",
    "weather_temperature_2m_max",
    "weather_precipitation_sum",
    "weather_wind_speed_10m_mean",
    "weather_wind_speed_10m_max",
    "weather_relative_humidity_2m_mean",
]

df_final = df_step_geography.merge(
    weather_daily[weather_columns],
    on="movement_date",
    how="left",
    validate="m:1",
)

inspect_dataframe(df_final, "Dataset final integrado")


## 13. Resumo final por POI

Este resumo final permite observar de forma sintetica a cobertura do dataset por POI antes das etapas de pre-processamento e analise exploratoria.

In [ ]:

final_poi_summary = (
    df_final.groupby(["geography_key", "poi_name"], dropna=False)
    .agg(
        rows=("movement_date", "size"),
        min_date=("movement_date", "min"),
        max_date=("movement_date", "max"),
        total_incoming_sum=("total_incoming", "sum"),
        avg_temperature=("weather_temperature_2m_mean", "mean"),
        total_precipitation=("weather_precipitation_sum", "sum"),
    )
    .reset_index()
    .sort_values(["rows", "total_incoming_sum"], ascending=[False, False])
)
final_poi_summary


## 14. Validacao final do dataset integrado

Nesta ultima etapa, sao verificados varios aspetos de consistencia do dataset final, de modo a garantir que este se encontra em condicoes adequadas para ser entregue aos colegas responsaveis pelo pre-processamento e pela analise exploratoria.

In [ ]:

print("Final dataset shape:", df_final.shape)
print("Duplicate column names:", int(df_final.columns.duplicated().sum()))
print("Duplicated rows by movement_date + geography_key:", int(df_final.duplicated(subset=["movement_date", "geography_key"]).sum()))
print("Missing target values:", int(df_final["total_incoming"].isna().sum()))
print("Number of POIs in final dataset:", df_final["geography_key"].nunique())
print("Date range:", df_final["movement_date"].min(), "to", df_final["movement_date"].max())

missing_summary = df_final.isna().sum().sort_values(ascending=False).to_frame("missing_values")
display(missing_summary.head(20))
display(df_final.head())


## 15. Exportacao do dataset final

O dataframe final e exportado para ficheiro CSV, de forma a facilitar a continuidade do trabalho nas etapas seguintes do projeto.

In [ ]:
output_path = DATA_DIR / "df_final_integrado_multi_poi_com_meteorologia.csv"

df_final.to_csv(output_path, index=False)

print("Dataset final integrado exportado com sucesso para a pasta central:")
print(output_path)
print(f"Dimensões finais do ficheiro: {df_final.shape}")


## Handoff para as proximas etapas

Este notebook termina na fase de integracao de dados. A partir deste ponto, o trabalho pode ser dividido entre **pre-processamento** e **analise exploratoria**, utilizando o ficheiro exportado e o dataframe `df_final`.

### Orientacoes para pre-processamento

Colunas mais diretamente relevantes para a etapa seguinte:

- Variavel alvo: `total_incoming`
- Chaves e identificacao: `movement_date`, `geography_key`, `poi_name`
- Variaveis temporais: `day_of_week`, `day_name`, `day_of_month`, `day_of_year`, `week_of_year`, `month_number`, `month_name`, `quarter_number`, `year_number`, `season`, `is_weekend`, `is_holiday`, `is_peak_season`, `holiday_name`, `is_business_day`
- Variaveis geograficas: `geography_type`, `geography_name`, `poi_id`, `poi_name`, `poi_category`, `timezone`, `is_sensorized`
- Variaveis meteorologicas: `weather_temperature_2m_mean`, `weather_temperature_2m_min`, `weather_temperature_2m_max`, `weather_precipitation_sum`, `weather_wind_speed_10m_mean`, `weather_wind_speed_10m_max`, `weather_relative_humidity_2m_mean`, `weather_hourly_records`
- Metadados da target: `incoming_record_count`, `incoming_source_rows`

Sugestoes para a etapa de pre-processamento:

- Converter `movement_date` para formato temporal e ordenar o dataset por `geography_key` e `movement_date`
- Criar lags e medias moveis por POI, por exemplo `lag_1`, `lag_7`, `rolling_mean_3` e `rolling_mean_7`
- Verificar se os valores em falta nas colunas temporais resultam da ausencia de correspondencia na `DIM_DATE`
- Decidir se colunas geograficas constantes por POI devem ser mantidas, codificadas ou removidas
- Avaliar a necessidade de normalizacao ou transformacao das variaveis meteorologicas

### Colunas que merecem atencao especial

No dataset final, algumas colunas apresentam muitos missing values:

- `island_name`, `concelho_name`, `freguesia_name`, `latitude`, `longitude`
- Parte das variaveis de calendario pode apresentar valores em falta em algumas datas devido a ausencia de correspondencia na `DIM_DATE`
- `Grota do Inferno` foi definida inicialmente no projeto, mas nao aparece na target deste subconjunto de dados

Estas colunas nao foram removidas nesta fase, uma vez que podem continuar a ser uteis para exploracao, validacao ou decisoes posteriores de limpeza.

### Orientacoes para analise exploratoria

Sugestoes para a etapa de EDA:

- Comparar a distribuicao de `total_incoming` entre os diferentes POIs
- Analisar a evolucao temporal da target por POI
- Explorar a relacao entre `total_incoming` e variaveis meteorologicas como temperatura, precipitacao e vento
- Identificar dias com valores anormalmente baixos ou elevados
- Explorar diferencas entre dias de semana, fins de semana e feriados
- Verificar a cobertura temporal de cada POI antes de retirar conclusoes comparativas

### Observacoes importantes

- A meteorologia integrada representa o contexto meteorologico diario da ilha de Sao Miguel, e nao observacoes especificas de cada POI
- O merge meteorologico foi realizado por `movement_date`, replicando o mesmo contexto diario para todos os POIs presentes em cada data
- Este notebook nao contempla preprocessamento, imputacao, encoding nem modelacao


## Nota final

Este notebook foca-se exclusivamente na integracao de dados. As etapas seguintes, incluindo criacao de lags, medias moveis, tratamento de missing values, encoding e modelacao, devem ser desenvolvidas em notebooks separados.